# Datová analytika - vytvoření sázkových kurzů
## Cíl

Úkolem bude získat výsledky NHL z veřejně dostupných zdrojů a následně určit počáteční sázkové kurzy.

Čtyři hlavní částí tohoto notebooku:

- **1. Stažení dat** – stáhnutí dat ze stránky Scrape This Site (část 1),

- **2. Zpracování dat** – analýza HTML a připrava data pro analýzu (část 2),

- **3. Analýza dat** – explorační analýzu dat (část 3),

- **4. Výsledek** – počáteční sázkové kurzy (část 4).

---

## Požadované knihovny

- **requests** – stažení HTML obsahu stránek se zápasy,

- **BeautifulSoup** – zpracování nestrukturovaných dat (HTML kódu) do tabulární podoby (DataSet),

- **Pandas** – provádění transformací dat,

- **Matplotlib** – prezentaci výsledků.

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
from glob import glob
import json
from glob import glob

## 1. Stažení dat
- Stažení tabulek do html souborů

In [ ]:
# URL
url = 'https://www.scrapethissite.com/pages/forms/?per_page=100'

try:
    # Vytvor request
    response = requests.get(url)
    response.raise_for_status()

    # BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')
    url_list = [i.text.strip() for i in soup.select(
        '.pagination a:not([aria-label="Next"])')]

    for index in url_list:
        table_url = f"https://www.scrapethissite.com/pages/forms/?page_num={index}&per_page=100"
        table_response = requests.get(table_url)
        table_response.raise_for_status()

        table_soup = BeautifulSoup(table_response.text,'html.parser')

        with open(f"./data/raw/hockey_table{index.zfill(2)}.html", "w", encoding="utf-8") as hpage:
            hpage.write(table_response.text)

        sleep(1)

except requests.exceptions.HTTPError as err:
    print(f"HTTP chyba nastala: {err}")
except Exception as err:
    print(f"Jiná chyba: {err}")
    

### Shrnutí
 
Stažení surových dat ze zdroje snížilo riziko problémů vyplývajících z aktualizací stránek během procesu extrakce. Tato metoda má i další výhodu: umožňuje snadný přístup k datům v jejich původní podobě, což je klíčové v případě nutnosti opětovného zpracování.
 
V dalším kroku se zaměříme na extrakci potřebných informací z `html` stránek, což je nezbytné pro provedení analýzy.

## 2. Zpracování dat
Uložení informací(`/data/raw`) o hokejových týmech do formátu JSON.. Extrahovaná data budou zahrnovat:

- Název týmu (`Team Name`),

- Rok (`Year`),

- Počet výher (`Wins`),

- Počet proher (`Losses`),

- Počet proher v prodloužení (`OT Losses` – Overtime Losses),

- Procento výher (`Win %`),

- Počet vstřelených gólů (`Goals For (GF)`),

- Počet inkasovaných gólů (`Goals Against (GA)`),

- Rozdíl skóre (`+ / -`).

Každý získaný záznam by měl být uspořádán do slovníku ve struktuře uvedené níže a poté přidán do seznamu výsledků:

```python
{
    'Team Name': 'Boston Bruins',
    'Year': '1990',
    'Wins': '44',
    'Losses': '24',
    'OT Losses': '',
    'Win %': '0.55',
    'Goals For (GF)': '299',
    'Goals Against (GA)': '264',
    '+ / -': '35'
}
```


In [ ]:
files = glob("./data/raw/*.html")

column_names = []
final_team_list = []

with open(files[0],"r") as f:
    soup = BeautifulSoup(f.read(),"html.parser")
    column_names = [item.text.strip() for item in soup.select(".table th")]

for file in files:
    team_list = []
    with open(file, "r") as f:
        soup = BeautifulSoup(f.read(),'html.parser')
        team_list = [team for team in soup.select("tr.team")]
        
        for team in team_list:
            team_values = [team_value.text.strip() for team_value in team.select("td")]
            team_dict = {}
            for column_name, value in zip(column_names, team_values):
                team_dict[column_name] = value

            final_team_list.append(team_dict)

with open("./data/raw/hockey_teams.json","w",encoding="utf-8") as f:
    json.dump(final_team_list, f, indent=4)